# GLM-5.2 determinism at temperature=0 across serving stacks

Sends the **same prompt** through **GLM-5.2** on three serving stacks
— Fireworks, Baseten, and Together-via-OpenRouter — `N` times each at
`temperature=0`, then asks: **how many of the `N` responses are identical
within each provider, and do the three stacks agree with each other?**

The Together leg reaches Together *through* OpenRouter's provider routing
(`provider.order: ["together"]`, no fallbacks) rather than our direct Together
API key, which is currently blocked.

The prompt is a **17-way customer-support intent classification** task with a
chain-of-thought twist: the model emits a JSON object with `reasoning` first
and the `label` second (enforced via `response_format: json_schema` where the
serving stack supports it). The reasoning gives nondeterminism real surface
area to show up — a single-token label is trivially deterministic, but a
free-text reasoning string is where temp=0 nondeterminism typically appears.

Determinism is reported on **three levels**:
- **full text** — exact raw JSON string equality (catches even whitespace/key-order drift),
- **reasoning** — just the reasoning field,
- **label** — just the classified intent (often stable even when reasoning varies).

This is *not* an eval — no grading, no accuracy. It's a pure determinism probe:
at temp=0 we'd hope for (near) identical outputs run-to-run on the same stack,
and it's informative (not guaranteed) whether different serving stacks of the
same model weights produce the same tokens.

Notes:
- `temperature=0` is requested; some stacks still have minor nondeterminism
  from batching/quantization/scheduling, so perfect agreement isn't assured.
- Baseten is routed via litellm's generic `openai/` provider against its
  OpenAI-compatible Model API endpoint (the `baseten/` prefix mis-routes on
  some litellm versions). Fireworks and OpenRouter use their native prefixes.
- Reasoning effort is **off** here (`thinking` disabled / no `reasoning_effort`)
  so we compare the raw greedy-decoded answer, not thinking traces — thinking
  can amplify nondeterminism and varies in length run-to-run.

In [1]:
# Run once if imports fail (uses the notebook kernel's Python).
import sys
!{sys.executable} -m pip install -q -e "../../.[eval]"


In [2]:
# --- edit these ---
FIREWORKS_MODEL = "fireworks_ai/accounts/fireworks/models/glm-5p2"
# Together account is currently blocked (403 request_blocked) when called
# directly, so we reach Together *through* OpenRouter instead. OpenRouter's
# `provider.order` forces the request to Together's backing endpoint, and
# `allow_fallbacks: false` guarantees it won't silently route elsewhere
# (configured in _kwargs_for below).
OPENROUTER_MODEL = "openrouter/z-ai/glm-5.2"
# Routed via litellm's generic `openai/` provider (see throughput_load_sweep
# notebook for why we avoid the `baseten/` prefix).
BASETEN_MODEL   = "openai/zai-org/GLM-5.2"
BASETEN_API_BASE = "https://inference.baseten.co/v1"

PROVIDERS = {
    "GLM-5.2 (Baseten)":            BASETEN_MODEL,
    "GLM-5.2 (Fireworks)":          FIREWORKS_MODEL,
    "GLM-5.2 (Together via OR)":    OPENROUTER_MODEL,
}

N = 100                     # times to send the prompt through each provider
TEMPERATURE = 0.0
MAX_TOKENS = 512            # room for reasoning + label (structured JSON output)
REQUEST_TIMEOUT_S = 120
CONCURRENCY_LEVEL = 2

# 17-way customer-support intent classification. The model emits a JSON object
# with reasoning first, then the label — so the output has real surface area
# for run-to-run nondeterminism (not just a single token).
INTENT_LABELS = [
    "billing_card_issue",
    "billing_chargeback",
    "billing_plan_change",
    "payment_failed",
    "payment_method_update",
    "account_login_issue",
    "account_password_reset",
    "account_signup",
    "account_close",
    "billing_refund",
    "account_profile_update",
    "order_status",
    "order_cancellation",
    "order_modification",
    "shipping_tracking",
    "shipping_address_change",
    "technical_bug",
    "feature_request",
]

CUSTOMER_MESSAGE = (
    "Hi, I was just charged twice for my monthly subscription this morning — "
    "the same amount came out of my card two times about ten minutes apart. "
    "I'd like the duplicate charge refunded to my original payment method."
)

# Prompt instructs JSON output with reasoning then label. Providers that
# support OpenAI-style json_schema structured output (Fireworks, OpenRouter/
# Together) additionally get a response_format schema that enforces the shape
# and the label enum; Baseten's Model API ignores response_format, so it gets
# the instructed-JSON path only (the model complies — verified).
PROMPT = (
    "You are a customer-support router. First reason briefly about the "
    "customer's issue, then classify the message into exactly one intent.\n\n"
    "Respond with ONLY a JSON object of this exact shape, no other text:\n"
    '{"reasoning": "<one or two sentences of reasoning>", "label": "<label>"}\n\n'
    "Allowed labels:\n"
    + "\n".join(f"- {label}" for label in INTENT_LABELS)
    + "\n\nCustomer message:\n"
    f'"""\n{CUSTOMER_MESSAGE}\n"""'
)

# JSON schema used as response_format for providers that support it.
JSON_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "intent_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "reasoning": {"type": "string"},
                "label": {"type": "string", "enum": INTENT_LABELS},
            },
            "required": ["reasoning", "label"],
            "additionalProperties": False,
        },
    },
}

# Providers whose serving stack honors response_format: json_schema.
_SUPPORTS_JSON_SCHEMA = {FIREWORKS_MODEL, OPENROUTER_MODEL}

In [3]:
import asyncio
import os
from collections import Counter
from pathlib import Path

import litellm
from dotenv import load_dotenv
from tqdm.auto import tqdm

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)
load_dotenv(training_dir / ".env")

missing = [k for k in ("FIREWORKS_API_KEY", "OPENROUTER_API_KEY", "BASETEN_API_KEY")
           if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing {missing}. Set them in {training_dir / '.env'} or your shell.")

litellm.drop_params = False
litellm.set_verbose = False
# litellm._turn_on_debug()  # uncomment only if you need to debug a real failure

MESSAGES = [{"role": "user", "content": PROMPT}]


def _kwargs_for(model: str) -> dict:
    kw: dict = {"temperature": TEMPERATURE}
    if model == BASETEN_MODEL:
        kw["api_base"] = BASETEN_API_BASE
        kw["api_key"] = os.getenv("BASETEN_API_KEY")
        kw["extra_body"] = {"thinking": {"type": "disabled"}}
    elif model == OPENROUTER_MODEL:
        # OpenRouter: disable reasoning via the `reasoning` map so we compare
        # greedy-decoded labels, not thinking traces. Force the backing
        # provider to Together (our direct Together account is blocked, so we
        # reach Together through OpenRouter's provider relationship instead).
        kw["extra_body"] = {
            "reasoning": {"enabled": False},
            "provider": {"order": ["together"], "allow_fallbacks": False},
        }
    else:
        # Fireworks: disable thinking via extra_body (top-level field).
        kw["extra_body"] = {"thinking": {"type": "disabled"}}
    # Enforce the JSON shape via response_format where the serving stack
    # supports it (Fireworks, OpenRouter). Baseten ignores response_format and
    # instead complies with the JSON instruction in PROMPT.
    if model in _SUPPORTS_JSON_SCHEMA:
        kw["response_format"] = JSON_SCHEMA
    return kw


def _parse_json_response(text: str) -> dict:
    """Extract reasoning + label from the model's JSON output. Lenient about
    leading/trailing whitespace or fenced code blocks."""
    import json as _json
    s = text.strip()
    # Strip ```json ... ``` fences if present.
    if s.startswith("```"):
        s = s.split("```", 2)[1]
        if s.startswith("json"):
            s = s[4:]
        s = s.strip()
    return _json.loads(s)


async def one_call(model: str, sem: asyncio.Semaphore) -> dict:
    async with sem:
        try:
            resp = await litellm.acompletion(
                model=model,
                messages=MESSAGES,
                max_tokens=MAX_TOKENS,
                timeout=REQUEST_TIMEOUT_S,
                **_kwargs_for(model),
            )
            text = (resp.choices[0].message.content or "").strip()
            try:
                parsed = _parse_json_response(text)
                reasoning = str(parsed.get("reasoning", "")).strip()
                label = str(parsed.get("label", "")).strip()
            except Exception:
                parsed = None
                reasoning = ""
                label = ""
            return {"ok": True, "text": text, "reasoning": reasoning, "label": label}
        except Exception as e:  # noqa: BLE001
            return {"ok": False, "error": repr(e)[:200]}

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Fire the same prompt N times at each provider

All `N` calls per provider run concurrently (limited by a small semaphore to
avoid hammering rate limits). We collect the raw response text for each call.

In [4]:
results: dict[str, list[dict]] = {}

async def run_provider(name: str, model: str) -> None:
    sem = asyncio.Semaphore(CONCURRENCY_LEVEL)
    tasks = [asyncio.ensure_future(one_call(model, sem)) for _ in range(N)]
    out: list[dict] = []
    with tqdm(total=N, desc=name, unit="req") as pbar:
        for fut in asyncio.as_completed(tasks):
            out.append(await fut)
            pbar.update(1)
    results[name] = out
    ok = [r for r in out if r["ok"]]
    errs = [r for r in out if not r["ok"]]
    print(f"{name:24} ok={len(ok)}/{N}  err={len(errs)}")
    if errs:
        print(f"  first error: {errs[0]['error']}")

for name, model in PROVIDERS.items():
    print(f"--- {name} ({model}) ---")
    await run_provider(name, model)

--- GLM-5.2 (Baseten) (openai/zai-org/GLM-5.2) ---


GLM-5.2 (Baseten): 100%|██████████| 100/100 [00:34<00:00,  2.94req/s]


GLM-5.2 (Baseten)        ok=100/100  err=0
--- GLM-5.2 (Fireworks) (fireworks_ai/accounts/fireworks/models/glm-5p2) ---


GLM-5.2 (Fireworks): 100%|██████████| 100/100 [01:12<00:00,  1.38req/s]


GLM-5.2 (Fireworks)      ok=100/100  err=0
--- GLM-5.2 (Together via OR) (openrouter/z-ai/glm-5.2) ---


GLM-5.2 (Together via OR): 100%|██████████| 100/100 [00:49<00:00,  2.00req/s]

GLM-5.2 (Together via OR) ok=100/100  err=0


## How many responses match within each provider?

For each provider, find the **modal** (most common) response and report how
many of the `N` calls produced it — on three levels: full JSON text, the
reasoning field, and the label. A provider is "deterministic at temp=0" if
the modal count equals `N` on all three.

In [5]:
per_provider_summary = {}
for name, rows in results.items():
    ok_rows = [r for r in rows if r["ok"]]
    n_ok = len(ok_rows)
    # Compare on three levels: full JSON text, reasoning only, label only.
    text_counts = Counter(r["text"] for r in ok_rows).most_common()
    reasoning_counts = Counter(r["reasoning"] for r in ok_rows).most_common()
    label_counts = Counter(r["label"] for r in ok_rows).most_common()
    modal_text, modal_text_count = text_counts[0] if text_counts else ("", 0)
    modal_reasoning, modal_reasoning_count = reasoning_counts[0] if reasoning_counts else ("", 0)
    modal_label, modal_label_count = label_counts[0] if label_counts else ("", 0)
    per_provider_summary[name] = {
        "n_ok": n_ok,
        "n_unique_responses": len(text_counts),
        "n_unique_reasoning": len(reasoning_counts),
        "n_unique_labels": len(label_counts),
        "modal_text": modal_text,
        "modal_text_count": modal_text_count,
        "modal_reasoning": modal_reasoning,
        "modal_reasoning_count": modal_reasoning_count,
        "modal_label": modal_label,
        "modal_label_count": modal_label_count,
    }
    print(f"\n=== {name} ===")
    print(f"  ok responses:      {n_ok}/{N}")
    print(f"  unique full text:  {len(text_counts)}   agreement: {modal_text_count}/{n_ok}" +
          (f"  ({modal_text_count/n_ok:.0%})" if n_ok else ""))
    print(f"  unique reasoning:  {len(reasoning_counts)}   agreement: {modal_reasoning_count}/{n_ok}" +
          (f"  ({modal_reasoning_count/n_ok:.0%})" if n_ok else ""))
    print(f"  unique labels:     {len(label_counts)}   agreement: {modal_label_count}/{n_ok}" +
          (f"  ({modal_label_count/n_ok:.0%})" if n_ok else ""))
    print(f"  modal label:       {modal_label!r}")
    print(f"  modal reasoning:")
    print("    " + modal_reasoning.replace("\n", "\n    "))


=== GLM-5.2 (Baseten) ===
  ok responses:      100/100
  unique full text:  11   agreement: 53/100  (53%)
  unique reasoning:  11   agreement: 53/100  (53%)
  unique labels:     1   agreement: 100/100  (100%)
  modal label:       'billing_refund'
  modal reasoning:
    The customer is reporting a duplicate charge for their subscription and is requesting that the extra amount be returned to their original payment method.

=== GLM-5.2 (Fireworks) ===
  ok responses:      100/100
  unique full text:  7   agreement: 44/100  (44%)
  unique reasoning:  7   agreement: 44/100  (44%)
  unique labels:     1   agreement: 100/100  (100%)
  modal label:       'billing_refund'
  modal reasoning:
    The customer is reporting a duplicate subscription charge and explicitly requesting that the extra amount be returned to their original payment method.

=== GLM-5.2 (Together via OR) ===
  ok responses:      100/100
  unique full text:  13   agreement: 26/100  (26%)
  unique reasoning:  13   agreement: 

## Do the three stacks agree with each other?

Take each provider's modal response and check whether all three match — i.e.
does the same model weight produce the same greedy output regardless of who
serves it?

In [6]:
from itertools import combinations

providers_ok = {name: s for name, s in per_provider_summary.items() if s["n_ok"]}
names = list(providers_ok)

print("Modal label per provider:")
for name in names:
    print(f"  {name:24} -> {providers_ok[name]['modal_label']!r}")

print("\n--- cross-provider agreement ---")
if len(names) >= 2:
    for field, label in (("modal_label", "label"),
                         ("modal_reasoning", "reasoning"),
                         ("modal_text", "full text")):
        print(f"\n  on {label}:")
        for a, b in combinations(names, 2):
            same = providers_ok[a][field] == providers_ok[b][field]
            print(f"    {a}  vs  {b}:  {'IDENTICAL' if same else 'DIFFERENT'}")
        all_same = len({providers_ok[n][field] for n in names}) == 1
        print(f"    all three agree on {label}: {all_same}")
else:
    print("  (not enough providers with successful responses to compare)")

Modal label per provider:
  GLM-5.2 (Baseten)        -> 'billing_refund'
  GLM-5.2 (Fireworks)      -> 'billing_refund'
  GLM-5.2 (Together via OR) -> 'billing_refund'

--- cross-provider agreement ---

  on label:
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Fireworks):  IDENTICAL
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Together via OR):  IDENTICAL
    GLM-5.2 (Fireworks)  vs  GLM-5.2 (Together via OR):  IDENTICAL
    all three agree on label: True

  on reasoning:
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Fireworks):  DIFFERENT
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Together via OR):  DIFFERENT
    GLM-5.2 (Fireworks)  vs  GLM-5.2 (Together via OR):  DIFFERENT
    all three agree on reasoning: False

  on full text:
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Fireworks):  DIFFERENT
    GLM-5.2 (Baseten)  vs  GLM-5.2 (Together via OR):  DIFFERENT
    GLM-5.2 (Fireworks)  vs  GLM-5.2 (Together via OR):  DIFFERENT
    all three agree on full text: False


## Inspect the disagreement

If a provider had more than one unique response, show the distinct variants
side by side so you can see *how* it diverged (typically a single token or
a trailing space / formatting difference).

In [7]:
for name, rows in results.items():
    ok_rows = [r for r in rows if r["ok"]]
    text_common = Counter(r["text"] for r in ok_rows).most_common()
    reasoning_common = Counter(r["reasoning"] for r in ok_rows).most_common()
    label_common = Counter(r["label"] for r in ok_rows).most_common()
    print(f"\n=== {name} ===")
    print(f"  unique full text: {len(text_common)} | unique reasoning: {len(reasoning_common)} | unique labels: {len(label_common)}")
    # Show ALL reasoning variants in full (no truncation) — this is the
    # interesting part for CoT determinism.
    print(f"  --- {len(reasoning_common)} distinct reasoning string(s) ---")
    for i, (reasoning, cnt) in enumerate(reasoning_common, 1):
        print(f"\n    variant {i}  (seen {cnt}x):")
        print("      " + reasoning.replace("\n", "\n      "))
    # If text varies but reasoning is identical, show the full-text variants
    # (likely JSON key ordering / whitespace differences).
    if len(reasoning_common) <= 1 and len(text_common) > 1:
        print(f"\n  reasoning identical, but full text has {len(text_common)} variants:")
        for i, (text, cnt) in enumerate(text_common, 1):
            print(f"\n    variant {i}  (seen {cnt}x):")
            print("      " + text.replace("\n", "\n      "))


=== GLM-5.2 (Baseten) ===
  unique full text: 11 | unique reasoning: 11 | unique labels: 1
  --- 11 distinct reasoning string(s) ---

    variant 1  (seen 53x):
      The customer is reporting a duplicate charge for their subscription and is requesting that the extra amount be returned to their original payment method.

    variant 2  (seen 17x):
      The customer is reporting a duplicate subscription charge and explicitly requesting that the extra amount be returned to their original payment method.

    variant 3  (seen 10x):
      The customer is reporting a duplicate charge for a subscription and is explicitly requesting that the extra amount be returned to their original payment method.

    variant 4  (seen 6x):
      The customer is reporting a duplicate charge for their subscription and is requesting that the extra charge be returned to their original payment method.

    variant 5  (seen 5x):
      The customer is reporting a duplicate charge for their subscription and is ex

In [8]:
# Compact summary table
print(f"{'provider':24} {'ok':>6} {'unq_text':>9} {'unq_reas':>9} {'unq_lbl':>8} {'lbl_agree':>10}")
print("-" * 70)
for name, s in per_provider_summary.items():
    lbl = f"{s['modal_label_count']}/{s['n_ok']}" if s["n_ok"] else "n/a"
    print(f"{name:24} {s['n_ok']:>3}/{N:<2} {s['n_unique_responses']:>9} "
          f"{s['n_unique_reasoning']:>9} {s['n_unique_labels']:>8} {lbl:>10}")

provider                     ok  unq_text  unq_reas  unq_lbl  lbl_agree
----------------------------------------------------------------------
GLM-5.2 (Baseten)        100/100        11        11        1    100/100
GLM-5.2 (Fireworks)      100/100         7         7        1    100/100
GLM-5.2 (Together via OR) 100/100        13        13        1    100/100
